# Accelerating LLM Inference: Speculative Decoding and Distillation - Part III

In this notebook, you'll learn how to:
1. Map the five main **model acceleration** techniques to this notebook series
2. Run **speculative decoding** in vLLM with a draft model and with n-gram lookup, then measure speedup and acceptance rate
3. **Distill** Qwen3-1.7B (teacher) into Qwen3-0.6B (student) with logit-based knowledge distillation
4. Serve the distilled student with vLLM

## Where Each Technique Lives

| Technique | What it reduces | Where |
|:--|:--|:--|
| **Quantization** | Weight bytes read per token (memory-bound decode) | [model-optimizations.ipynb](model-optimizations.ipynb) |
| **Continuous batching** | Idle GPU slots between requests | [model-inference.ipynb](model-inference.ipynb) |
| **Caching** (KV cache, prefix cache) | Repeated prefill compute and KV memory waste | [model-inference.ipynb](model-inference.ipynb) |
| **Speculative decoding** | Number of sequential target forward passes | This notebook, Part 1 |
| **Distillation** | Model size needed for a given quality | This notebook, Part 2 |

Benchmarking the combined result with GuideLLM and `lm_eval` is covered in [model-benchmark.ipynb](model-benchmark.ipynb).

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, gc, json, math, subprocess, sys, time
import requests

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs("outputs", exist_ok=True)

TARGET = "Qwen/Qwen3-1.7B"   # target model (swap for Qwen/Qwen3-4B on a larger GPU)
DRAFT = "Qwen/Qwen3-0.6B"    # draft model, shares the Qwen3 tokenizer
PORT = 8001

print(f"Target: {TARGET}\nDraft:  {DRAFT}")

> **Version assumption:** the speculative decoding configuration below uses the `speculative_config` JSON dict (`vllm serve --speculative-config '{...}'`), which replaced the older `--speculative-model` / `--num-speculative-tokens` flags. This repo only pins `vllm>=0.6` (`pyproject.toml`), so the notebook assumes a recent release (roughly 0.10 or later). The `draft_model` method is not available in every V1-engine release; if vLLM rejects it, use `ngram` or an EAGLE head instead. Metric names (`vllm:spec_decode_*`) are also version dependent. Check against the docs for the installed version.
>
> This notebook was written without access to a GPU host, so the vLLM cells were not executed in the authoring environment.

# Part 1: Speculative Decoding

## What Is Speculative Decoding?

Autoregressive decoding is **memory-bound**: each new token needs one full forward pass of the target model, and that pass mostly waits on weight loads. Speculative decoding trades spare compute for fewer sequential passes:

1. A cheap **proposer** drafts `k` tokens.
2. The **target** model scores all `k` positions in a single forward pass.
3. Drafted tokens are accepted left to right while they match what the target would have sampled; the first mismatch is replaced by the target's own token.

Every step emits at least one target-quality token, and up to `k + 1`. With rejection sampling the output distribution is identical to plain decoding (lossless). With greedy decoding the text should match, up to numerical noise.

The speedup depends on the **acceptance rate** and the proposer's cost:

`mean accepted length = 1 + accepted_tokens / num_drafts`

| Proposer | Needs | Works best when |
|:--|:--|:--|
| **Draft model** | A smaller model with the same tokenizer | Draft agrees with target on typical text |
| **N-gram (prompt lookup)** | Nothing: matches the last n tokens against the prompt and output so far | Output copies from the prompt: summarization, extraction, code edits, RAG |

## Helpers: Launch, Query, Measure

Unlike [model-inference.ipynb](model-inference.ipynb), speculative decoding is a server startup option, so each configuration needs its own server process. The helper below starts `vllm serve` on a separate port, waits for it, and stops it afterwards. Run this on a CUDA host with vLLM installed.

In [ ]:
VLLM_URL = f"http://localhost:{PORT}"

def launch_vllm(model, spec=None, log="outputs/vllm.log"):
    cmd = ["vllm", "serve", model, "--port", str(PORT),
           "--dtype=bfloat16", "--max-model-len", "4096"]
    if spec:
        cmd += ["--speculative-config", json.dumps(spec)]
    proc = subprocess.Popen(cmd, stdout=open(log, "w"), stderr=subprocess.STDOUT)
    for attempt in range(120):
        if proc.poll() is not None:
            raise RuntimeError(f"vLLM exited early, see {log}")
        try:
            if requests.get(f"{VLLM_URL}/v1/models", timeout=5).status_code == 200:
                return proc
        except requests.ConnectionError:
            pass
        time.sleep(5)
    proc.terminate()
    raise RuntimeError("vLLM server not reachable after 10 minutes.")

def stop_vllm(proc):
    proc.terminate()
    proc.wait(timeout=60)
    time.sleep(5)  # let GPU memory free up

In [ ]:
from openai import OpenAI
client = OpenAI(base_url=f"{VLLM_URL}/v1", api_key="unused")

def get_counters(prefix="vllm:spec_decode"):
    """Scrape /metrics and sum samples per metric name (labels dropped)."""
    out = {}
    for line in requests.get(f"{VLLM_URL}/metrics").text.split("\n"):
        if line.startswith("#") or not line.startswith(prefix):
            continue
        name = line.split("{")[0].split()[0]
        out[name] = out.get(name, 0.0) + float(line.split()[-1])
    return out

The prompt set mixes open-ended generation with prompts whose answers copy from the context. N-gram lookup only helps on the second kind, so the split makes the difference visible.

In [ ]:
DOC = (
    "vLLM schedules requests at the token level. When a request finishes, its "
    "slot is refilled immediately. KV cache memory is split into fixed-size "
    "blocks, so requests of different lengths share memory without "
    "fragmentation. Prefix caching reuses blocks across requests that share a "
    "prompt prefix."
)

PROMPTS = [
    # open-ended: low n-gram overlap
    "Explain why LLM decoding is memory-bound in three sentences.",
    "Write a short product description for a noise-cancelling headset.",
    "What are the trade-offs between latency and throughput in serving?",
    # copy-heavy: output repeats the prompt
    f"Repeat the following text exactly, word for word:\n\n{DOC}",
    f"Quote the sentence about prefix caching from this text:\n\n{DOC}",
    f"Rewrite this text as a bullet list, keeping the original wording:\n\n{DOC}",
]

def run_benchmark(model, max_tokens=128):
    rows = []
    for p in PROMPTS:
        t0 = time.time()
        r = client.chat.completions.create(
            model=model, messages=[{"role": "user", "content": p}],
            max_tokens=max_tokens, temperature=0.0,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        dt = time.time() - t0
        n = r.usage.completion_tokens
        rows.append({"text": r.choices[0].message.content, "tokens": n,
                     "latency": dt, "tok_s": n / dt})
    return rows

def summarize(name, rows, before, after):
    tok = sum(r["tokens"] for r in rows)
    lat = sum(r["latency"] for r in rows)
    d = {k: after.get(k, 0) - before.get(k, 0) for k in after}
    drafts = d.get("vllm:spec_decode_num_drafts_total", 0)
    drafted = d.get("vllm:spec_decode_num_draft_tokens_total", 0)
    accepted = d.get("vllm:spec_decode_num_accepted_tokens_total", 0)
    return {
        "name": name, "tok_s": tok / lat, "latency": lat / len(rows),
        "accept_rate": accepted / drafted if drafted else None,
        "accept_len": 1 + accepted / drafts if drafts else None,
    }

## Baseline: Speculative Decoding Off

In [ ]:
results, texts = [], {}

proc = launch_vllm(TARGET)
rows = run_benchmark(TARGET)       # first pass warms up CUDA graphs and kernels
before = get_counters()
rows = run_benchmark(TARGET)
results.append(summarize("baseline", rows, before, get_counters()))
texts["baseline"] = [r["text"] for r in rows]
stop_vllm(proc)

print(results[-1])

## Draft Model

`num_speculative_tokens` is `k`. Larger `k` raises the ceiling on tokens per step but lowers the acceptance of later positions, so there is a sweet spot, typically 3 to 5 for a draft model.

```python
{"method": "draft_model", "model": "Qwen/Qwen3-0.6B", "num_speculative_tokens": 4}
```

The draft model runs `k` sequential passes per step, so its cost matters. A 0.6B draft for a 1.7B target is a poor ratio (draft is about 35% of the target). Gains grow with target size: 0.6B drafting for 8B or larger is a more realistic setup. Qwen3-1.7B is used here so the notebook fits a small GPU.

In [ ]:
def run_config(name, spec):
    proc = launch_vllm(TARGET, spec)
    run_benchmark(TARGET)
    before = get_counters()
    rows = run_benchmark(TARGET)
    res = summarize(name, rows, before, get_counters())
    stop_vllm(proc)
    texts[name] = [r["text"] for r in rows]
    return res

results.append(run_config("draft-model k=4", {
    "method": "draft_model", "model": DRAFT, "num_speculative_tokens": 4,
}))
print(results[-1])

## N-gram (Prompt Lookup)

No second model and no extra GPU memory. The proposer searches the existing context for the last `prompt_lookup_max` to `prompt_lookup_min` tokens and proposes whatever followed the match. If nothing matches, no tokens are drafted and decoding proceeds normally, so the overhead is small.

In [ ]:
results.append(run_config("ngram k=4", {
    "method": "ngram", "num_speculative_tokens": 4,
    "prompt_lookup_max": 4, "prompt_lookup_min": 2,
}))
print(results[-1])

## Compare

In [ ]:
base = results[0]
print(f"{'config':<18}{'tok/s':>8}{'latency(s)':>12}{'speedup':>9}{'accept':>9}{'tok/step':>10}")
for r in results:
    acc = f"{r['accept_rate']:.0%}" if r["accept_rate"] is not None else "-"
    ln = f"{r['accept_len']:.2f}" if r["accept_len"] is not None else "-"
    print(f"{r['name']:<18}{r['tok_s']:>8.1f}{r['latency']:>12.2f}"
          f"{r['tok_s'] / base['tok_s']:>8.2f}x{acc:>9}{ln:>10}")

for name, t in texts.items():
    same = sum(a == b for a, b in zip(t, texts['baseline']))
    print(f"{name}: {same}/{len(t)} outputs identical to baseline (greedy)")

**Reading the numbers**

- **accept**: fraction of drafted tokens the target accepted (`accepted_tokens / draft_tokens`).
- **tok/step**: mean tokens emitted per target forward pass (`1 + accepted / drafts`). This is the upper bound on speedup before proposer overhead.
- Expect n-gram to win clearly on the copy-heavy prompts and do nothing on the open-ended ones. Running the two groups separately (`PROMPTS[:3]` vs `PROMPTS[3:]`) shows this directly.
- Speculative decoding helps most at **low batch size**, where decode is memory-bound. At high concurrency the GPU is already compute-saturated and extra verification work can reduce throughput. Re-run the comparison with concurrent requests (see [model-inference.ipynb](model-inference.ipynb)) before enabling it in production.
- Sampling with `temperature > 0` lowers acceptance relative to greedy.

# Part 2: Distillation

## What Is Distillation?

Distillation trains a small **student** to imitate a larger **teacher**. Unlike quantization, which compresses a fixed architecture, distillation produces a different, cheaper model. Unlike plain SFT on hard labels, the student sees the teacher's full next-token distribution, which carries more signal per token (relative likelihood of wrong-but-plausible tokens).

Logit-based KD loss:

`L = alpha * T^2 * KL(softmax(z_t / T) || softmax(z_s / T)) + (1 - alpha) * CE(z_s, y)`

- `T` softens both distributions; `T^2` keeps gradient magnitude comparable across temperatures.
- `alpha` mixes teacher signal with ground-truth cross-entropy.
- Teacher and student must share a vocabulary. Qwen3-1.7B and Qwen3-0.6B do.

This section uses a tiny slice of WikiText-2 so it runs on a laptop in a few minutes. It demonstrates the mechanics, not a production recipe: real runs use far more tokens, and distilled models are typically trained on teacher-generated or domain data.

## Setup

In [ ]:
import random
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

TEACHER_ID = "Qwen/Qwen3-1.7B"
STUDENT_ID = "Qwen/Qwen3-0.6B"
SAVE_DIR = "../models/Qwen3-0.6B-distilled"

SEQ_LEN = 128        # tokens per training block
BATCH = 4
STEPS = 100
LR = 1e-5
TEMP = 2.0
ALPHA = 0.5          # weight of the KD term
N_TRAIN_BLOCKS = 400
N_EVAL_BLOCKS = 32

device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(0); random.seed(0)
print(f"Device: {device}")

> **Memory:** the student trains in fp32 (about 2.4 GB weights, plus gradients and Adam state, roughly 10 GB total). The teacher stays frozen in bf16 (about 3.4 GB). A 16 GB laptop is enough with the settings above. Lower `BATCH` or `SEQ_LEN` if you run out.

## Data

Text is concatenated, tokenized once, and cut into fixed-length blocks. Evaluation blocks come from the validation split so they are held out from training.

In [ ]:
tok = AutoTokenizer.from_pretrained(STUDENT_ID)

def make_blocks(split, n_blocks):
    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split=split)
    text = "\n\n".join(t for t in ds["text"] if t.strip())
    ids = tok(text[: n_blocks * SEQ_LEN * 8], return_tensors="pt").input_ids[0]
    n = min(n_blocks, ids.numel() // SEQ_LEN)
    return ids[: n * SEQ_LEN].view(n, SEQ_LEN)

train_blocks = make_blocks("train", N_TRAIN_BLOCKS)
eval_blocks = make_blocks("validation", N_EVAL_BLOCKS)
print(f"train: {tuple(train_blocks.shape)}  eval: {tuple(eval_blocks.shape)}")

## Load Teacher and Student

In [ ]:
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_ID, dtype=torch.bfloat16).to(device).eval()
for p in teacher.parameters():
    p.requires_grad_(False)

student = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID, dtype=torch.float32).to(device)

assert teacher.config.vocab_size == student.config.vocab_size
print(f"teacher {sum(p.numel() for p in teacher.parameters())/1e9:.2f}B  "
      f"student {sum(p.numel() for p in student.parameters())/1e9:.2f}B params")

## Evaluate Before Training

Two metrics on held-out blocks:

- **Perplexity** of the student on the text (lower is better).
- **Teacher agreement**: fraction of positions where the student's top-1 next token equals the teacher's, and the mean KL from teacher to student (lower KL is better).

In [ ]:
@torch.no_grad()
def evaluate(student, teacher, blocks, bs=4):
    student.eval()
    nll, n_tok, agree, kl_sum = 0.0, 0, 0, 0.0
    for i in range(0, len(blocks), bs):
        x = blocks[i:i + bs].to(device)
        s = student(x).logits[:, :-1].float()
        t = teacher(x).logits[:, :-1].float()
        y = x[:, 1:]
        nll += F.cross_entropy(s.reshape(-1, s.size(-1)), y.reshape(-1),
                               reduction="sum").item()
        n_tok += y.numel()
        agree += (s.argmax(-1) == t.argmax(-1)).sum().item()
        kl_sum += F.kl_div(F.log_softmax(s, -1), F.log_softmax(t, -1),
                           log_target=True, reduction="sum").item()
    return {"ppl": math.exp(nll / n_tok), "agree": agree / n_tok,
            "kl": kl_sum / n_tok}

before = evaluate(student, teacher, eval_blocks)
print("student before:", {k: round(v, 4) for k, v in before.items()})

## KD Loss

In [ ]:
def kd_loss(s_logits, t_logits, labels, temp=TEMP, alpha=ALPHA):
    s = s_logits[:, :-1].float()
    t = t_logits[:, :-1].float()
    y = labels[:, 1:]
    kd = F.kl_div(
        F.log_softmax(s / temp, -1), F.log_softmax(t / temp, -1),
        log_target=True, reduction="none",
    ).sum(-1).mean() * temp ** 2
    ce = F.cross_entropy(s.reshape(-1, s.size(-1)), y.reshape(-1))
    return alpha * kd + (1 - alpha) * ce, kd.item(), ce.item()

## Train

In [ ]:
opt = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.0)
student.train()
student.config.use_cache = False

t0 = time.time()
for step in range(1, STEPS + 1):
    idx = torch.randint(0, len(train_blocks), (BATCH,))
    x = train_blocks[idx].to(device)
    with torch.no_grad():
        t_logits = teacher(x).logits
    s_logits = student(x).logits
    loss, kd, ce = kd_loss(s_logits, t_logits, x)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
    opt.step(); opt.zero_grad(set_to_none=True)
    if step == 1 or step % 10 == 0:
        print(f"step {step:>4}/{STEPS}  loss {loss.item():.3f}  "
              f"kd {kd:.3f}  ce {ce:.3f}  ({time.time() - t0:.0f}s)")

## Evaluate After Training

In [ ]:
after = evaluate(student, teacher, eval_blocks)

print(f"{'metric':<22}{'before':>10}{'after':>10}")
print(f"{'perplexity':<22}{before['ppl']:>10.2f}{after['ppl']:>10.2f}")
print(f"{'teacher top-1 agree':<22}{before['agree']:>10.1%}{after['agree']:>10.1%}")
print(f"{'KL(teacher||student)':<22}{before['kl']:>10.4f}{after['kl']:>10.4f}")

> **Note:** with a few hundred steps on a small slice, expect KL and agreement to improve modestly. Perplexity on WikiText can move either way: the student is being pulled toward the teacher, not toward the reference text, and Qwen3 was trained on far more data than this slice. Treat the before/after comparison as a check that the training loop works, not as a quality result.

## Save the Student

In [ ]:
student.to(torch.bfloat16).save_pretrained(SAVE_DIR)
tok.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")

## Serve the Distilled Student with vLLM

The saved directory is a standard Hugging Face checkpoint, so vLLM serves it like any other model. Free the training models first, then launch:

```bash
vllm serve ../models/Qwen3-0.6B-distilled --dtype=bfloat16 --max-model-len 4096
```

In [ ]:
del teacher, student, opt
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

proc = launch_vllm(SAVE_DIR)
resp = client.chat.completions.create(
    model=SAVE_DIR,
    messages=[{"role": "user", "content": "What is speculative decoding?"}],
    max_tokens=80, temperature=0.0,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print(resp.choices[0].message.content)
stop_vllm(proc)

The distilled student can also serve as the **draft model** in Part 1: `{"method": "draft_model", "model": SAVE_DIR, ...}`. Distilling a draft toward the target's output distribution raises the acceptance rate, which is the main lever on speculative decoding speedup.

## Where Distillation Fits in the Pipeline

Distillation is a **training-stage** technique. It produces new weights, so it sits next to the other training steps in `src/vmp/training/` and upstream of quantization and serving:

```
data -> SFT (sft.py) -> preference tuning (dpo.py) -> merge (merge.py)
     -> distillation (teacher -> smaller student)
     -> quantization (llm-compressor)
     -> vLLM serving (batching, caching, speculative decoding)
```

- **SFT / DPO** adapt behavior of a fixed-size model with LoRA adapters. **Distillation** changes the model size while preserving behavior.
- Typical order: fine-tune or align the large teacher first, then distill into the student, then quantize the student, then serve with speculative decoding if latency at low batch size matters.
- Each stage is evaluated with the same harness ([model-benchmark.ipynb](model-benchmark.ipynb)) so quality and latency regressions are visible per step.

## Summary

In this notebook, you:

- Mapped the five acceleration techniques to the notebooks that cover them
- Ran **speculative decoding** in vLLM with a draft model and with n-gram lookup, and measured tokens/s, latency, acceptance rate, and tokens per step
- Saw why speedup depends on acceptance rate, proposer cost, and batch size
- Trained a Qwen3-0.6B student against a Qwen3-1.7B teacher with **logit-based KD** (temperature-scaled KL plus CE) and compared student metrics before and after
- Served the distilled student with vLLM and placed distillation in the training pipeline

## Resources

- [vLLM Speculative Decoding](https://docs.vllm.ai/en/latest/features/spec_decode.html)
- [Fast Inference from Transformers via Speculative Decoding (Leviathan et al.)](https://arxiv.org/abs/2211.17192)
- [Accelerating LLM Decoding with Speculative Sampling (Chen et al.)](https://arxiv.org/abs/2302.01318)
- [Distilling the Knowledge in a Neural Network (Hinton et al.)](https://arxiv.org/abs/1503.02531)
- [Qwen3 on Hugging Face](https://huggingface.co/Qwen)
- [vLLM Github](https://github.com/vllm-project/vllm)